In [0]:
%run ./03_Gold_Modeling

** Dim_Product**

In [0]:
# Load product
df_product = load_silver_table("product")

In [0]:
# Define product key as surrogate key
df_product = df_product.withColumnRenamed("product_key", "sur_product_key")

# Write in gold schema
write_gold_table(df_product , "dim_product")

**Dim_reseller**

In [0]:
# Load reseller
df_reseller = load_silver_table("reseller")

In [0]:
# Define reseller key as surrogate key
df_reseller = df_reseller.withColumnRenamed("reseller_key", "sur_reseller_key")

# Write in gold schema
write_gold_table(df_reseller, "dim_reseller")

**Dim_region**

In [0]:
# Load region
df_region = load_silver_table("region")

In [0]:
# Define region key as surrogate key
df_region = df_region.withColumnRenamed("sales_territory_key", "sur_territory_key")

# Write in gold schema
write_gold_table(df_region, "dim_region")

**Dim_Bridge**

In [0]:
df_bridge = spark.sql("""
    SELECT 
        employee_key AS sur_employee_key,
        sales_territory_key AS sur_territory_key
    FROM project.silver.salesperson_region
""")
write_gold_table(df_bridge, "dim_bridge")


**Fact Sales**

In [0]:
load_silver_table("sales")

In [0]:
fact_sales = spark.sql("""
    SELECT 
        s.sales_order_number,
        dd.date_key AS sur_order_date_key,  
        dp.sur_product_key,                
        dr.sur_reseller_key,               
        dsp.sur_employee_key,              
        dt.sur_territory_key,             
        s.quantity,
        s.unit_price_amount AS unit_price,
        s.sales_amount,
        s.cost_amount AS total_cost,
        (s.sales_amount - s.cost_amount) AS margin_amount,
        s.currency_code AS currency
    FROM project.silver.sales s

    JOIN project.gold.dim_date dd 
        ON s.order_date = dd.date
    JOIN project.gold.dim_product dp 
        ON s.product_key = dp.sur_product_key
    JOIN project.gold.dim_reseller dr 
        ON s.reseller_key = dr.sur_reseller_key
    JOIN project.gold.dim_salesperson dsp 
        ON s.employee_key = dsp.sur_employee_key
    JOIN project.gold.dim_region dt 
        ON s.sales_territory_key = dt.sur_territory_key
""")

# Write fact_sales to gold schema
write_gold_table(fact_sales, "fact_sales")

In [0]:
%sql
-- Check if any sales records were lost during the Gold transformation
SELECT 
    (SELECT COUNT(*) FROM project.silver.sales) AS silver_sales_count,
    (SELECT COUNT(*) FROM project.gold.fact_sales) AS gold_sales_count,
    (SELECT COUNT(*) FROM project.silver.sales) - (SELECT COUNT(*) FROM project.gold.fact_sales) AS difference

In [0]:
%sql
-- Check for duplicate sales orders in the Gold layer
SELECT sales_order_number, COUNT(*)
FROM project.gold.fact_sales
GROUP BY sales_order_number
HAVING COUNT(*) > 1

In [0]:
%sql
--This query validates that our Gold Layer is ready for Business Intelligence. It joins our sales facts with our product dimensions to calculate key performance indicators (KPIs) like Total Revenue and Profit Margin per Category, ordered by the highest performing segments."

-- Total Revenue and Profit Margin by Product Category
-- This query validates the integration between fact_sales and dim_product
SELECT 
    p.category, 
    SUM(s.sales_amount) AS total_revenue,
    SUM(s.margin_amount) AS total_profit,
    ROUND((SUM(s.margin_amount) / SUM(s.sales_amount)) * 100, 2) AS profit_margin_pct
FROM project.gold.fact_sales s
JOIN project.gold.dim_product p ON s.sur_product_key = p.sur_product_key
GROUP BY p.category
ORDER BY total_revenue DESC